CleanSIAPMonthlydata.py -- jsayre@ucdavis.edu

2023-11-16: When monthly data is yearly, the new_harv or new_planted variable is 0. This is probably what we want.

In [ ]:
import pandas as pd
import os
import numpy as np

### Directories
base_dir                =  os.path.join(os.path.expanduser("~"), "Dropbox", "Projects", "Maize_prediction")
output_dir              =  os.path.join(base_dir, "Data",  "SIAP_monthly", "Output")
inegi_mun_dir           =  os.path.join(base_dir, "Data",  "muncodes/")
joel_dir                =  os.path.join(base_dir, "Data")
plot_dir                =  os.path.join(base_dir, "plots", "monthly_prod/")

### Inputs
monthly_siap_dta        =  os.path.join(output_dir,"mnthly_siap.dta")
max_harv_mth_dta        =  os.path.join(output_dir,"max_harv_mnth.dta")
max_harv_mth_by_yr_dta  =  os.path.join(output_dir,"max_harv_mnth_by_year.dta")
planting_months         =  os.path.join(joel_dir,  "planting_months_harmonic_regression.csv")
ndvi_peaks              =  os.path.join(joel_dir,  "muni_ndvi_peak_dates.csv")
ndvi_troughs            =  os.path.join(joel_dir,  "muni_ndvi_min_dates.csv")

### Intermediates


### Outputs
comp_sat_plant           =  os.path.join(output_dir,"comp_sat_plant.csv")

### Functions
def create_new_columns(row):
    if row['Irrig'] == 1 and row['Cycle'] == 1:
        return pd.Series({'max_harv_irrig_fw': row['max_harv_month_median'], 'max_plants_irrig_fw': row['max_plants_month_median']})
    elif row['Irrig'] == 0 and row['Cycle'] == 1:
        return pd.Series({'max_harv_rf_fw': row['max_harv_month_median'], 'max_plants_rf_fw': row['max_plants_month_median']})
    elif row['Irrig'] == 1 and row['Cycle'] == 2:
        return pd.Series({'max_harv_irrig_spsm': row['max_harv_month_median'], 'max_plants_irrig_spsm': row['max_plants_month_median']})
    elif row['Irrig'] == 0 and row['Cycle'] == 2:
        return pd.Series({'max_harv_rf_spsm': row['max_harv_month_median'], 'max_plants_rf_spsm': row['max_plants_month_median']})
    else:
        return pd.Series()
    
def compute_distance(row, iterate_cols = ['max_plants_irrig_fw', 'max_plants_irrig_spsm', 'max_plants_rf_fw', 'max_plants_rf_spsm'],
                     sat_month = 'sat_month'):
    distances = {}
    for col in iterate_cols:
        if pd.isna(row[col]):
            distances[col] = np.nan
        else:
            distances[col] = min(abs(row[sat_month] - row[col]), 12 - abs(row[sat_month] - row[col]))
    min_distance = min([val for val in distances.values() if not pd.isna(val)])
    if pd.isna(min_distance):
        return None
    tie_cases = [col for col, distance in distances.items() if distance == min_distance]
    if len(tie_cases) == 1:
        return (min_distance, tie_cases[0])
    else:
        return (min_distance, tie_cases)
    
def replace_season_id(row):
    if isinstance(row['season_id'], list):
        if 'max_plants_irrig_spsm' in row['season_id'] and 'max_plants_rf_spsm' in row['season_id']:
            return 'SPSM'
        elif 'max_plants_irrig_fw' in row['season_id'] and 'max_plants_rf_fw' in row['season_id']:
            return 'FW'
        elif 'max_plants_irrig_spsm' in row['season_id'] and 'max_plants_irrig_fw' in row['season_id']:
            return 'Irrig'
        elif 'max_plants_rf_spsm' in row['season_id'] and 'max_plants_rf_fw' in row['season_id']:
            return 'RF'
    else:
        if row['season_id'] == 'max_plants_irrig_fw':
            return 'Irrig, FW'
        elif row['season_id'] == 'max_plants_irrig_spsm':
            return 'Irrig, SPSM'
        elif row['season_id'] == 'max_plants_rf_fw':
            return 'RF, FW'
        elif row['season_id'] == 'max_plants_rf_spsm':
            return 'RF, SPSM'
        else:
            return None

In [ ]:
### Read in NDVI peaks and troughs from landsat
ndvi_peak_df = pd.read_csv(ndvi_peaks)
ndvi_peak_df['muncode'] = ndvi_peak_df['CVE_ENT'].astype(str).str.zfill(2) + ndvi_peak_df['CVE_MUN'].astype(str).str.zfill(3)
ndvi_peak_df['month_float'] = (ndvi_peak_df['peak_day_of_year']/30.4166666666667)+1
ndvi_peak_df['closest_month_ndvi_peak'] = ndvi_peak_df['month_float'].round(0)#.astype(int)
ndvi_peak_df = ndvi_peak_df.drop(['Unnamed: 0','CVE_ENT','CVE_MUN','peak_day_of_year','peak_date','month_float'],axis=1)
ndvi_peak_df = ndvi_peak_df[ndvi_peak_df['year'] > 2017]

ndvi_min_df = pd.read_csv(ndvi_troughs)
ndvi_min_df['muncode'] = ndvi_min_df['CVE_ENT'].astype(int).astype(str).str.zfill(2) + ndvi_min_df['CVE_MUN'].astype(int).astype(str).str.zfill(3)
ndvi_min_df['month_float'] = (ndvi_min_df['t']/30.4166666666667)+1
ndvi_min_df['closest_month_ndvi_min'] = ndvi_min_df['month_float'].round(0)#.astype(int)
ndvi_min_df = ndvi_min_df.drop(['CVE_ENT','CVE_MUN','t','Date','month_float','_merge'],axis=1)
ndvi_min_df = ndvi_min_df[ndvi_min_df['year']> 2017]

ndvi_df = ndvi_peak_df.merge(ndvi_min_df, on = ['muncode','year'], how = 'outer')

In [ ]:
## Read in max harvest/ planting month
max_harv_yr_df = pd.read_stata(max_harv_mth_by_yr_dta)
max_harv_yr_df = max_harv_yr_df[max_harv_yr_df['Crop'] == 'Maíz grano'].drop(['Crop'],axis=1)
max_harv_yr_df = max_harv_yr_df[max_harv_yr_df['year'] != 2023]
max_harv_yr_df = max_harv_yr_df[['Irrig','Cycle','year','muncode','max_harv_month','max_plants_month']]
max_harv_yr_df = max_harv_yr_df.pivot_table(index=['muncode','year'], columns=['Irrig', 'Cycle'], values=['max_harv_month','max_plants_month']).reset_index()
max_harv_yr_df.columns = ['muncode','year', 'max_harv_month_rf_fw', 'max_harv_month_rf_sp', 'max_harv_month_irrig_fw', 'max_harv_month_irrig_sp',  'max_plants_month_rf_fw', 'max_plants_month_rf_sp', 'max_plants_month_irrig_fw', 'max_plants_month_irrig_sp']
max_harv_yr_df
## Read in monthly data aggregated by year to compute likely season we should get peak for
month_df = pd.read_stata(monthly_siap_dta)
month_df = month_df[month_df['Crop'] == 'Maíz grano'].drop(['Crop'],axis=1)
month_df = month_df[month_df['year'] != 2023]
month_df = month_df[['Irrig','Cycle','year','Mes','muncode','ha_planted']]
month_df = month_df.groupby(['Irrig','Cycle','muncode','year']).sum().reset_index()
month_df = month_df.pivot_table(index=['muncode','year'], columns=['Irrig', 'Cycle'], values='ha_planted').reset_index()
month_df.columns = ['muncode','year', 'ha_planted_irrig_fw', 'ha_planted_irrig_sp', 'ha_planted_rf_fw', 'ha_planted_rf_sp']
month_df['total'] = month_df[['ha_planted_irrig_fw', 'ha_planted_irrig_sp', 'ha_planted_rf_fw', 'ha_planted_rf_sp']].sum(axis=1)
for share_var in ['ha_planted_irrig_fw', 'ha_planted_irrig_sp', 'ha_planted_rf_fw', 'ha_planted_rf_sp']:
    month_df[share_var] = month_df[share_var] / month_df['total']
month_df = month_df.drop('total',axis=1)

max_ha_planted = month_df[['ha_planted_irrig_fw', 'ha_planted_irrig_sp', 'ha_planted_rf_fw', 'ha_planted_rf_sp']].idxmax(axis=1)
max_ha_planted = max_ha_planted.replace({'ha_planted_irrig_fw': 'irrig_fw', 'ha_planted_irrig_sp': 'irrig_spsm', 'ha_planted_rf_fw': 'rf_fw', 'ha_planted_rf_sp': 'rf_spsm'})
month_df['biggest_szn'] = max_ha_planted
month_df = month_df.dropna(subset=['biggest_szn'])
month_df = month_df[['muncode','year','biggest_szn']]

### Merge these all together
max_harv_yr_df = max_harv_yr_df.merge(month_df, on=['muncode','year'], how='left')
max_harv_yr_df = max_harv_yr_df.merge(ndvi_df, on=['muncode','year'], how='inner')
harv_cols = ['max_harv_month_rf_fw', 'max_harv_month_rf_sp', 'max_harv_month_irrig_fw', 'max_harv_month_irrig_sp']
plant_cols = ['max_plants_month_rf_fw', 'max_plants_month_rf_sp', 'max_plants_month_irrig_fw', 'max_plants_month_irrig_sp']

max_harv_yr_df['how_close_planted'] = max_harv_yr_df.dropna(subset=plant_cols, how='all').dropna(subset='closest_month_ndvi_min').apply(compute_distance, iterate_cols = plant_cols, sat_month='closest_month_ndvi_min', axis=1).apply(lambda x: x[0])
max_harv_yr_df['season_id_planted'] = max_harv_yr_df.dropna(subset=plant_cols, how='all').dropna(subset='closest_month_ndvi_min').apply(compute_distance, iterate_cols = plant_cols, sat_month='closest_month_ndvi_min', axis=1).apply(lambda x: x[1])

max_harv_yr_df['actual_gap_planted'] = max_harv_yr_df.dropna(subset=plant_cols, how='all').dropna(subset='closest_month_ndvi_min').apply(lambda row: row[row['season_id_planted'][0]]-row['closest_month_ndvi_min'] if isinstance(row['season_id_planted'],list) else row[row['season_id_planted']]-row['closest_month_ndvi_min'], axis=1)

max_harv_yr_df['how_close_harvest'] = max_harv_yr_df.dropna(subset=harv_cols, how='all').dropna(subset='closest_month_ndvi_peak').apply(compute_distance, iterate_cols = harv_cols, sat_month='closest_month_ndvi_peak', axis=1).apply(lambda x: x[0])
max_harv_yr_df['season_id_harvest'] = max_harv_yr_df.dropna(subset=harv_cols, how='all').dropna(subset='closest_month_ndvi_peak').apply(compute_distance, iterate_cols = harv_cols, sat_month='closest_month_ndvi_peak', axis=1).apply(lambda x: x[1])

max_harv_yr_df['actual_gap_harvest'] = max_harv_yr_df.dropna(subset=harv_cols, how='all').dropna(subset='closest_month_ndvi_peak').apply(lambda row: row[row['season_id_harvest'][0]]-row['closest_month_ndvi_peak'] if isinstance(row['season_id_harvest'],list) else row[row['season_id_harvest']]-row['closest_month_ndvi_peak'], axis=1)

In [ ]:
max_harv_yr_df['actual_gap_planted'].value_counts()

In [ ]:
max_harv_yr_df['biggest_szn_rf'] = max_harv_yr_df['biggest_szn'].str.contains('rf')
max_harv_yr_df['biggest_szn_sp'] = max_harv_yr_df['biggest_szn'].str.contains('sp')
max_harv_yr_df['closest_szn_rf_planted'] = max_harv_yr_df['season_id_planted'].astype(str).str.contains('rf')
max_harv_yr_df['closest_szn_sp_planted'] = max_harv_yr_df['season_id_planted'].astype(str).str.contains('sp')
max_harv_yr_df['closest_szn_rf_harvest'] = max_harv_yr_df['season_id_harvest'].astype(str).str.contains('rf')
max_harv_yr_df['closest_szn_sp_harvest'] = max_harv_yr_df['season_id_harvest'].astype(str).str.contains('sp')

percentage_rf_plant = (max_harv_yr_df['biggest_szn_rf'] == max_harv_yr_df['closest_szn_rf_planted']).mean() * 100
percentage_sp_plant = (max_harv_yr_df['biggest_szn_sp'] == max_harv_yr_df['closest_szn_sp_planted']).mean() * 100
percentage_rf_harv  = (max_harv_yr_df['biggest_szn_rf'] == max_harv_yr_df['closest_szn_rf_harvest']).mean() * 100
percentage_sp_harv  = (max_harv_yr_df['biggest_szn_sp'] == max_harv_yr_df['closest_szn_sp_harvest']).mean() * 100

percentage_rf_plant, percentage_sp_plant, percentage_rf_harv, percentage_sp_harv


In [ ]:


### Read in harmonic mean predicted planting month
sat_month_df = pd.read_csv(planting_months)
sat_month_df['muncode'] = sat_month_df['CVE_ENT'].astype(int).astype(str).str.zfill(2) + sat_month_df['CVE_MUN'].astype(int).astype(str).str.zfill(3)
sat_month_df = sat_month_df[['muncode','planting_month']]
sat_month_df.columns = ['muncode','sat_month']
sat_month_df = sat_month_df.merge(month_df, on=['muncode','sat_month'], how='left')
def compute_largest_var(row):
    values = row[['ha_planted_irrig_fw', 'ha_planted_irrig_sp', 'ha_planted_rf_fw', 'ha_planted_rf_sp']]
    max_value = max(values.dropna(), default=np.nan)
    if pd.isna(max_value):
        return np.nan
    else:
        return list(values[values == max_value].index)[0]

sat_month_df['largest_season_planted'] = sat_month_df.apply(compute_largest_var, axis=1)
sat_month_df = sat_month_df.drop(['ha_planted_irrig_fw', 'ha_planted_irrig_sp', 'ha_planted_rf_fw', 'ha_planted_rf_sp'],axis=1)
sat_month_df['largest_season_planted'] = sat_month_df['largest_season_planted'].replace({'ha_planted_irrig_fw':'Irrig, FW',
                                                                                            'ha_planted_irrig_sp':'Irrig, SPSM',
                                                                                            'ha_planted_rf_fw':'RF, FW',
                                                                                            'ha_planted_rf_sp':'RF, SPSM'})

In [ ]:

### Read in max planting/harvesting months from monthly SIAP data
max_harv_df = pd.read_stata(max_harv_mth_dta)
max_harv_df = max_harv_df[max_harv_df['Crop'] == 'Maíz grano'].drop(['Crop'],axis=1)



max_harv_df[['max_harv_irrig_fw', 'max_harv_irrig_spsm', 'max_harv_rf_fw',
       'max_harv_rf_spsm', 'max_plants_irrig_fw', 'max_plants_irrig_spsm',
       'max_plants_rf_fw', 'max_plants_rf_spsm']] = max_harv_df.apply(create_new_columns, axis=1)
max_harv_df = max_harv_df.groupby(['muncode','Estado','Municipio']).mean()
max_harv_df = max_harv_df.reset_index().drop(['Irrig','Cycle','max_harv_month_median','max_plants_month_median'],axis=1)
max_harv_df = max_harv_df.drop(['max_harv_rf_fw','max_harv_rf_spsm','max_harv_irrig_fw','max_harv_irrig_spsm'],axis=1)
max_harv_df = max_harv_df.merge(sat_month_df, on='muncode', how='inner')

max_harv_df['how_close'] = max_harv_df.apply(compute_distance, axis=1).apply(lambda x: x[0])
max_harv_df['season_id'] = max_harv_df.apply(compute_distance, axis=1).apply(lambda x: x[1])

max_harv_df['season_id'] = max_harv_df.apply(replace_season_id, axis=1)

max_harv_df.to_csv(comp_sat_plant, index=False)

In [ ]:
max_harv_df[max_harv_df['how_close']>1]
max_harv_df['how_close'].value_counts()
